In [0]:
%pip install pdfplumber -q

%restart_python

In [0]:
import os

BASE_PATH = "/Volumes/dataplatform01_central_dev_catalog_01/bronze_raw_sap_bo/sapbo_webi/dev/object_type=pdf/ingestion_date=2026-06-01/"

# --- Step 1: collect all PDF file paths ---
pdf_files = []
for root, dirs, files in os.walk(BASE_PATH):
    # Skip hidden directories to avoid slowdowns/timeouts
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    for f in files:
        if f.upper().endswith(".PDF"):
            pdf_files.append(os.path.join(root, f))

pdf_files = sorted(pdf_files)
print(f"Total PDF files found: {len(pdf_files)}")

# --- Read raw content of each file ---
raw_files = {}   # filename -> list of raw lines
failed_files = []
for path in pdf_files:
    filename = os.path.basename(path)
    try:
        with open(path, "r", encoding="utf-8", errors="replace") as fh:
            raw_files[filename] = fh.readlines()
    except Exception as e:
        failed_files.append((filename, str(e)))
        print(f"  WARNING: skipped {filename} — {e}")

print(f"Files read into memory: {len(raw_files)}")
if failed_files:
    print(f"Files failed to read: {len(failed_files)}")

# --- Build files inventory DataFrame ---
import re
import pandas as pd

records = []
for path in pdf_files:
    filename = os.path.basename(path)
    match = re.search(r"(\d+)", filename)
    doc_id = match.group(1) if match else None
    records.append({"Report Type": "WEBI", "Report Document ID": doc_id})

df_files = pd.DataFrame(records)
print(f"\nFiles inventory ({len(df_files)} rows):")
display(df_files)

In [0]:
import pdfplumber
import re
import pandas as pd


def split_text_into_sections(text):
    """Split extracted page text into sections separated by one or more blank lines."""
    sections = []
    current = []
    for line in text.splitlines():
        if line.strip() == "":
            if current:
                sections.append("\n".join(current))
                current = []
        else:
            current.append(line)
    if current:
        sections.append("\n".join(current))
    return sections


# --- Extract content from each PDF ---
records = []
failed_pdfs = []

for path in [p for p in pdf_files if "14598" in os.path.basename(p)]:
    filename = os.path.basename(path)
    match = re.search(r"(\d+)", filename)
    doc_id = match.group(1) if match else None

    try:
        with pdfplumber.open(path) as pdf:
            total_pages = len(pdf.pages)
            section_counter = 0  # global section index across the whole document
            print(f"Processing {filename} ({total_pages} pages)")
            for page_num, page in enumerate(pdf.pages):
                text = page.extract_text() or ""

                print(f"  File text read: {len(text)} chars")
                page_sections = split_text_into_sections(text)

                for sec_idx, sec_text in enumerate(page_sections):
                    records.append({
                        "Report Type":        "WEBI",
                        "Report Document ID": doc_id,
                        "Filename":           filename,
                        "Total Pages":        total_pages,
                        "Page Number":        page_num + 1,        # 1-based
                        "Section Index":      section_counter,     # global across doc
                        "Section on Page":    sec_idx,             # local within page
                        "Section Text":       sec_text.strip(),
                        "Line Count":         len(sec_text.strip().splitlines()),
                    })
                    section_counter += 1

    except Exception as e:
        failed_pdfs.append((filename, str(e)))
        print(f"  WARNING: skipped {filename} — {e}")


df_pdf_sections = pd.DataFrame(records)
print(f"PDFs processed:          {len(pdf_files) - len(failed_pdfs)} / {len(pdf_files)}")
if failed_pdfs:
    print(f"PDFs failed:             {len(failed_pdfs)}")
print(f"Total sections extracted: {len(df_pdf_sections)}")

# Summary per document
summary = (
    df_pdf_sections
    .groupby(["Report Type", "Report Document ID", "Filename", "Total Pages"])
    .agg(Total_Sections=("Section Index", "count"))
    .reset_index()
)
print(f"\nDocument summary ({len(summary)} documents):")
display(summary)

display(df_pdf_sections)